| Pieza (Semana 6) | Fórmula | Cambio en este lab |
|---|---|---|
| `make_positional_encoding` | $PE(t,2i)=\sin(t/10000^{2i/d})$, $PE(t,2i+1)=\cos(t/10000^{2i/d})$ | igual, con `max_len = block_size = 128` |
| `layer_norm` | $\gamma \odot \frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta$ | igual |
| `masked_self_attention` | $\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$, con $M_{ij}=-\infty$ si $j>i$ | ahora procesa **batches** `(B, T, d_model)` |
| FFN | $\text{ReLU}(xW_1+b_1)W_2+b_2$ | misma fórmula, con $d_{ff}=4\,d_{model}=512$ (en S6 era $2\,d_{model}$) |
| Bloque decoder | Add & LayerNorm | se apilan **`n_layers = 3`** bloques (orden Pre-LN, ver Task 1.2) |
| LM head | $\text{logits}=xW_{lm}+b_{lm}$ | vocabulario de **65 caracteres** en lugar de palabras |
| `generate` | $\text{softmax}(z/\tau)$ + `multinomial` | se separa en 3 estrategias: greedy, top-k y top-p |

## Bloque 0: imports y semillas

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, os, random, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

SEED = 1337
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | device: {device}')

PyTorch 2.14.0+cpu | device: cpu


# Task 1

## Task 1.1: dataset, vocabulario y batches

In [ ]:
# Descarga del corpus tiny_shakespeare (~1.1M caracteres)
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
DATA_PATH = 'tiny_shakespeare.txt'
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(URL, DATA_PATH)

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Caracteres totales: {len(text):,}')
print(text[:250])

Caracteres totales: 1,115,394
----------------------------------------
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [3]:
# Vocabulario de caracteres unicos (ordenado para que los indices sean reproducibles)
chars = sorted(set(text))
vocab_size = len(chars)
assert vocab_size == 65, f'El vocabulario debe tener 65 caracteres, tiene {vocab_size}'

# Mappings bidireccionales caracter <-> indice
stoi = {ch: i for i, ch in enumerate(chars)}   # char  -> int
itos = {i: ch for ch, i in stoi.items()}       # int   -> char

encode = lambda s: [stoi[c] for c in s]          # str       -> list[int]
decode = lambda l: ''.join(itos[int(i)] for i in l)  # list[int] -> str

print(f'vocab_size = {vocab_size}')
print('Vocabulario:', repr(''.join(chars)))
print('encode("ROMEO:") =', encode('ROMEO:'))
print('decode(encode("ROMEO:")) =', decode(encode('ROMEO:')))

vocab_size = 65
Vocabulario: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
encode("ROMEO:") = [30, 27, 25, 17, 27, 10]
decode(encode("ROMEO:")) = ROMEO:


In [4]:
# Codificar todo el texto como secuencia de indices
data = torch.tensor(encode(text), dtype=torch.long)

# Split 90/10 respetando el orden temporal (sin shuffle): el bloque final del texto
# queda para validacion, asi el modelo nunca ve durante el entrenamiento el texto que se evalua.
n_train = int(0.9 * len(data))
train_data = data[:n_train]
val_data   = data[n_train:]
print(f'data: {data.shape} | train: {len(train_data):,} tokens | val: {len(val_data):,} tokens')

data: torch.Size([1115394]) | train: 1,003,854 tokens | val: 111,540 tokens


In [5]:
# Hiperparametros minimos del enunciado
block_size = 128   # T: largo de contexto
d_model    = 128
n_heads    = 4
n_layers   = 3
batch_size = 64
lr         = 3e-4
epochs     = 20    # el enunciado pide minimo 10. Con 20 se obtiene val ppl ~6.0 (con 10 da ~7.2)

d_k  = d_model // n_heads   # 32 por cabeza
d_ff = 4 * d_model          # 512, como en Vaswani et al.

def get_batch(split, batch_size=batch_size, block_size=block_size):
    '''
    Genera un batch de secuencias de largo fijo.
      x[b] = data[i : i+T]        (entrada)
      y[b] = data[i+1 : i+T+1]    (objetivo = entrada desplazada un paso al futuro)
    Asi en cada posicion t el modelo aprende P(x_{t+1} | x_1..x_t).
    '''
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(f'x: {tuple(xb.shape)}  y: {tuple(yb.shape)}')
print('x[0][:30] =', repr(decode(xb[0][:30])))
print('y[0][:30] =', repr(decode(yb[0][:30])))
assert torch.equal(xb[:, 1:], yb[:, :-1]), 'y debe ser x desplazado un paso'

x: (64, 128)  y: (64, 128)
x[0][:30] = ' if die, be brief,\nThat our sw'
y[0][:30] = 'if die, be brief,\nThat our swi'
